## C-Drama RAG: Data Cleaning 

### Cleaning Plan
- Scores → drop score < 7.5 
- Watchers → drop dramas with < 500 watchers
- Episodes → drop dramas with < 10 episodes
- Synopsis → strip junk text, drop if < 150 chars after stripping
- Genres & tags → lowercase and strip whitespace, drop if empty

In [10]:
import pandas as pd
from pathlib import Path
import json

RAW_PATH = Path("../data/raw/dramas.jsonl")
CLEAN_PATH = Path("../data/cleaned/dramas_clean.jsonl")

# pd.read_json + lines=True = NDJSON (each line is one JSON object)
raw_df = pd.read_json(RAW_PATH, lines=True)
print(f"Raw dataset: {len(raw_df)} dramas")

Raw dataset: 2710 dramas


### Step 1: Filter numeric columns

In [11]:
enough_episodes = raw_df["episodes"] >= 10
enough_watchers = raw_df["watchers"] >= 500
good_score = raw_df["mdl_score"].between(7.5, 10.0)

df = raw_df[enough_episodes & enough_watchers & good_score].copy()

print(f"After numeric filters: {len(df)} dramas")
print(f"Dropped: {len(raw_df) - len(df)} dramas")

# How many survivors are missing genres or tags?
missing_genres = (df["genres"].apply(len) == 0).sum()
missing_tags = (df["tags"].apply(len) == 0).sum()
print(f"Survivors with empty genres: {missing_genres}")
print(f"Survivors with empty tags:   {missing_tags}")

After numeric filters: 1467 dramas
Dropped: 1243 dramas
Survivors with empty genres: 1
Survivors with empty tags:   6


### Step 2: Drop dramas without genres or tags

In [12]:
before = len(df)
df = df[(df["genres"].apply(len) > 0) & (df["tags"].apply(len) > 0)].copy()
print(f"Dropped {before - len(df)} dramas with empty genres or tags")
print(f"Remaining: {len(df)} dramas")

Dropped 7 dramas with empty genres or tags
Remaining: 1460 dramas


### Step 3: Clean synopsis text

In [13]:
df["synopsis"] = (
    df["synopsis"]
    .str.replace(r"[\n\r\t]+", " ", regex=True)              # normalize whitespace first so $ anchors work below
    .str.replace(r"(?i)\s*\(Source:.*?\)", "", regex=True)   # strip (Source: MDL)
    .str.replace(r"(?i)\s*Adapted from.*$", "", regex=True)  # strip "Adapted from..." footnotes
    .str.replace(r"~+\s*", "", regex=True)                   # strip ~~~~ decorations
    .str.replace(r"Edit Translation.*$", "", regex=True)      # strip translation junk + language list
    .str.strip()
)

print("Synopsis cleaning done.")

Synopsis cleaning done.


In [14]:
for synopsis in df["synopsis"].sample(5, random_state=42).values:
    print(synopsis)
    print("-" * 80)

A top-secret assassination mission brings together the righteous martial artist Fu Xiao and the cold commander Yan Chang Yun, who initially clash. After a failure causes Fu Xiao to lose her memory, she becomes an innocent pawn in Yan's revenge plot. As they navigate the dangerous court and martial arts world, they uncover an old injustice and form a lifelong partnership.
--------------------------------------------------------------------------------
This centers around two descendants, Yuwen Tuo and Chen Jing Chou, and their plans to recover the Northern Zhou and Chen Dynasties, their respective heritages, by finding several mystical, powerful weapons. These weapons also help in closing up the doorway that demons use to enter and wreak havoc in the mortal world. An unlikely friendship forms, yet they are fated to be enemies.  Chen Jing Chou's adventures lead him to befriend alot of different people, such as Yu Xiao Xue and Tuoba Yu'er, while Yuwen Tuo meets Dugu Ning Ke, a supposedly 

### Step 4: Drop short synopses

In [15]:
before = len(df)

df = df[df["synopsis"].str.len() >= 150].copy()

print(f"Dropped {before - len(df)} dramas with synopsis < 150 chars")
print(f"Remaining: {len(df)} dramas")

Dropped 29 dramas with synopsis < 150 chars
Remaining: 1431 dramas


### Step 5: Normalize list columns

Meta-tags describe *how a drama was made* (source material, format, distribution) rather than *what it's about*.
Including them in the embedding context creates false content clusters — e.g. novel adaptations get pulled together regardless of genre or theme.

In [ ]:
from collections import Counter

# Inspect all tags so we can spot production/format meta-tags to strip.
# Lowercased + stripped here to match what the META_TAGS filter compares against.
all_tags = Counter(t.lower().strip() for tags in df["tags"] for t in tags)
print(f"Unique tags: {len(all_tags)}")
for tag, count in all_tags.most_common():
    print(f"{count:5d}  {tag}")

In [17]:
# Tags that describe production/format/source — not narrative content.
# These create false similarity clusters in the embedding space.
META_TAGS = {
    # source material
    "adapted from a novel",
    "adapted from a web novel",
    "adapted from a light novel",
    "adapted from a manga",
    "adapted from a manhua",
    "adapted from a manhwa",
    "adapted from a comic",
    "adapted from a webtoon",
    "based on a true story",
    "adapted from a game",
    "original work",
    # format / distribution
    "web series",
    "short length series",
    "filmed vertically",
    "mdl remake",
    # production notes
    "censored adaptation",
    "censored adaptation of same-sex original work",
}

# Lowercase + strip whitespace; filter meta-tags from tags only (genres vocab is controlled)
df["genres"] = df["genres"].apply(lambda lst: [item.lower().strip() for item in lst])
df["tags"] = df["tags"].apply(
    lambda lst: [t for t in (item.lower().strip() for item in lst) if t not in META_TAGS]
)

# Drop dramas left with no tags after meta-tag removal
before = len(df)
df = df[df["tags"].apply(len) > 0].copy()
print(f"Dropped {before - len(df)} dramas with empty tags after meta-tag filter")

print("Genres and tags normalized.")

# Sanity check: confirm none of the meta-tags survived
found = [t for tags in df["tags"] for t in tags if t in META_TAGS]
print(f"Remaining meta-tags after filter: {len(found)}")

Dropped 10 dramas with empty tags after meta-tag filter
Genres and tags normalized.
Remaining meta-tags after filter: 0


### Step 6: Verify the cleaned data

In [18]:
print(f"Score range: {df['mdl_score'].min()} – {df['mdl_score'].max()}")
print(f"Min episodes: {df['episodes'].min()}")
print(f"Min watchers: {df['watchers'].min()}")
print(f"Min synopsis length: {df['synopsis'].str.len().min()}")
print(f"Empty genres: {(df['genres'].apply(len) == 0).sum()}")  # .sum() counts True values
print(f"Empty tags: {(df['tags'].apply(len) == 0).sum()}")
print(f"Null synopses: {df['synopsis'].isna().sum()}")
print(f"Null years: {df['year'].isna().sum()}")

Score range: 7.5 – 9.1
Min episodes: 10
Min watchers: 502
Min synopsis length: 151
Empty genres: 0
Empty tags: 0
Null synopses: 0
Null years: 0


### Step 7: Final summary and save

In [19]:
print(f"Clean dataset: {len(df)} dramas")
print(f"Dropped: {len(raw_df) - len(df)} dramas total")

CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)

# Use stdlib json instead of pandas to_json — ujson (used internally by pandas)
# escapes forward slashes (/→\/), which bloats URLs and text fields unnecessarily.
with open(CLEAN_PATH, "w", encoding="utf-8") as f:
    for record in df.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved to {CLEAN_PATH}")

Clean dataset: 1421 dramas
Dropped: 1289 dramas total
Saved to ..\data\cleaned\dramas_clean.jsonl
